# Stage 1 Drift Run — The Real Measurement

**What this notebook does:** for a sample of real MATH-500 problems, runs BOTH conditions, matching Mirage (2506.04210)'s TTBC1 exactly:
- **Short**: true natural generation, no injection, no artificial ceiling — the model's own unmodified stopping point.
- **Long**: Wait injection (literal "Wait"), UNCAPPED — every genuine stop attempt gets intercepted, with only the model's real max generation length (24576) as the ceiling.

Then checks whether the answer flips between the two, and in which direction:
- **drift**: correct short -> wrong long (the phenomenon we're testing for).
- **reverse**: wrong short -> correct long (the mirror; common reverse flips would suggest noise, not directional drift).
- **stable_correct** / **stable_wrong**: no flip.
- **short_truncated** / **long_truncated**: the model never reached a boxed answer in that condition (reported honestly, not dropped).

**Section 5B runs all three models automatically** (`MODELS_TO_RUN` in that cell), one after another, MATH-500 only. Output filenames are tagged with the model automatically, so results never collide across models. Add a new model later by adding one line to `AVAILABLE_MODELS` in Section 2.5.

**This notebook auto-detects how many GPUs are actually available** and picks the right strategy: multi-GPU parallel workers if several are visible, or a single sequential run (with resume support) if only one is.

**Files to upload first:** `harness.py` (v0.3.4), `wait_injection.py`, `math500_matcher.py`, `drift_run_worker.py`. Run top to bottom.

## 1. Install packages
Let this finish completely before running anything else.

In [ ]:
!pip install -q numpy torch transformers accelerate datasets


## 1.5. GPU memory settings (set before anything else touches CUDA)

`expandable_segments` specifically targets *fragmentation*-driven allocation failures (small allocations failing even when total free memory looks sufficient) -- exactly the pattern in the OOM warnings we've been seeing. Must be set before any CUDA call, so it lives right after installs.

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("PYTORCH_CUDA_ALLOC_CONF set.")


## 1.6. Zombie kernel cleanup utility

**What this solves:** running this notebook repeatedly (crashes, restarts, re-runs) leaves old Jupyter kernels alive in the background, each still holding GPU memory for a model that finished loading but was never released, because the *kernel process itself* was never shut down, not because our code did anything wrong. This is what caused the `Process 214360 has 14.21 GiB` OOM earlier.

**How it works, and why it's safe:** it only ever considers a PID for killing if BOTH are true: (1) `ps` shows it as an `ipykernel_launcher` process **owned by your own account** — a process belonging to a teammate is invisible to your `ps` in the first place, so it can never be a candidate — and (2) `nvidia-smi` confirms it's actually holding GPU memory right now. Your **current, live kernel is checked by PID and always excluded**. Tested against realistic mock output before ever touching a real GPU: current kernel excluded, other users' processes excluded, real zombies correctly found.

**Runs automatically before every model load in Section 5B** (see below), and is also callable directly any time you suspect stale memory:
```python
cleanup_zombie_kernels()
```

In [ ]:
import os, subprocess, signal, time

def cleanup_zombie_kernels(verbose=True):
    """Kill any OTHER ipykernel processes under this account that are still
    holding GPU memory. Never touches the current kernel or another user's
    process -- see the markdown above for the exact safety conditions."""
    my_pid = os.getpid()
    my_user = subprocess.run(["whoami"], capture_output=True, text=True).stdout.strip()

    # PIDs actually holding GPU memory right now, per nvidia-smi (host-level,
    # so this can list PIDs from other users' sessions too -- that's fine,
    # we only ever act on the intersection with our own `ps` view below).
    smi = subprocess.run(
        ["nvidia-smi", "--query-compute-apps=pid,used_memory",
         "--format=csv,noheader,nounits"],
        capture_output=True, text=True
    ).stdout.strip()
    gpu_pids = {}
    for line in smi.splitlines():
        if not line.strip():
            continue
        pid_str, mem_str = [x.strip() for x in line.split(",")]
        gpu_pids[int(pid_str)] = int(mem_str)

    # Our own ipykernel processes. NOTE: ps truncates a long username to 8
    # chars + '+' by default (e.g. "jupyter+"), which would silently break
    # the ownership match -- user:32 avoids that.
    ps = subprocess.run(
        ["ps", "-eo", "pid,user:32,cmd"], capture_output=True, text=True
    ).stdout
    zombies = []
    for line in ps.splitlines()[1:]:
        parts = line.split(None, 2)
        if len(parts) < 3:
            continue
        pid_str, user, cmd = parts
        pid = int(pid_str)
        if (user == my_user and "ipykernel_launcher" in cmd
                and pid != my_pid and pid in gpu_pids):
            zombies.append((pid, gpu_pids[pid]))

    if not zombies:
        if verbose:
            print("  [cleanup] no zombie kernels found holding GPU memory.")
        return 0

    freed_mb = 0
    for pid, mem_mb in zombies:
        try:
            os.kill(pid, signal.SIGKILL)
            print(f"  [cleanup] killed zombie kernel PID {pid} (was holding {mem_mb} MiB)")
            freed_mb += mem_mb
        except (ProcessLookupError, PermissionError) as e:
            print(f"  [cleanup] could not kill PID {pid}: {e}")
    time.sleep(2)  # let CUDA actually release the memory before we proceed
    if verbose:
        print(f"  [cleanup] done, ~{freed_mb} MiB freed.")
    return freed_mb

# quick manual test -- safe to run any time, reports but doesn't require any model loaded
cleanup_zombie_kernels()


## 2. Version check — FIRST, every time

In [ ]:
import harness, wait_injection, math500_matcher, inspect, os
print("harness version:", harness.HARNESS_VERSION, "(want 0.3.5)")
print("wait_injection version:", getattr(wait_injection, "WAIT_INJECTION_VERSION", "MISSING -- STALE FILE"), "(want 0.3.0)")
print("Wait default phrasing:", repr(wait_injection.WAIT_TEXT), "(want 'Wait')")

wi_src = inspect.getsource(wait_injection)
has_uncapped_fix = "max_injections is not None and" in wi_src
print("wait_injection has uncapped-injection fix:", has_uncapped_fix, "(want True)")
if not has_uncapped_fix:
    print("  !! STOP: this file will crash with 'TypeError: >= not supported between "
          "int and NoneType' the moment it hits a genuine stop attempt. Re-upload "
          "wait_injection.py before running anything else.")

has_memory_fix = "output_scores=output_scores" in wi_src
print("wait_injection has output_scores memory fix:", has_memory_fix, "(want True)")
if not has_memory_fix:
    print("  !! STOP: this file will likely OOM on larger models at long budgets "
          "(unnecessary ~20GB score capture). Re-upload wait_injection.py.")

harness_src = inspect.getsource(harness)
has_harness_memory_fix = "output_scores=output_scores" in harness_src
print("harness has output_scores memory fix:", has_harness_memory_fix, "(want True)")

src = inspect.getsource(math500_matcher)
print("matcher has x-in-prefix fix:", "_VAR_IN_PREFIX" in src, "(want True)")
print("matcher has nested-frac fix:", "_FRAC_BRACED" in src, "(want True)")

print()
print("--- multi-dataset files (needed for the overnight GSM8K/AIME run) ---")
missing_files = []
for fname in ["gsm8k_matcher.py", "aime_matcher.py", "datasets_registry.py"]:
    if not os.path.exists(fname):
        missing_files.append(fname)
if missing_files:
    print(f"  !! STOP: missing file(s): {missing_files}. Upload them before running Section 5B,")
    print("     or the run will crash immediately on the first dataset that needs them.")
else:
    print("  gsm8k_matcher.py, aime_matcher.py, datasets_registry.py: all present.")
    try:
        import datasets_registry
        print("  datasets_registry imports cleanly. Registered datasets:", list(datasets_registry.DATASETS.keys()))
        for dk, cfg in datasets_registry.DATASETS.items():
            print(f"    {dk}: pool_size={cfg['pool_size']}")
    except Exception as e:
        print(f"  !! STOP: datasets_registry failed to import: {type(e).__name__}: {e}")

import torch
N_GPUS = torch.cuda.device_count()
print("GPUs visible to this notebook:", N_GPUS)
print("CUDA_VISIBLE_DEVICES (env):", os.environ.get("CUDA_VISIBLE_DEVICES", "(not set)"))
print()
if N_GPUS >= 2:
    print(f">> Will use the MULTI-GPU path (Section 5A) across {N_GPUS} GPUs.")
else:
    print(">> Will use the SINGLE-GPU sequential path (Section 5B), since only 1 GPU is visible.")


## 2.5. Choose the model

**One-line switch below: `MODEL_KEY`.** Everything downstream (both GPU paths, output filenames) picks it up automatically. Output files are tagged with the model, so no run ever overwrites another.

**Models** (all DeepSeek-R1-Distill family, same `<think>` convention): qwen-1.5b, qwen-7b, llama-8b. Dataset is MATH-500 only in this notebook (long budget 24576, matching DeepSeek's own max generation length and Mirage's TTBC1 ceiling).

Phi-4-reasoning and Qwen3-8B are NOT in the model list yet: they're different model families needing a compatibility fix + spot-check first (see ROADMAP.md, Phase 2), and adding them blindly would silently produce meaningless results.

**Note on the single-GPU path (5B):** it loops over ALL models in `MODELS_TO_RUN` (default: all three), running them back to back with per-model resume. `MODEL_KEY` above is only used by the multi-GPU path (5A) and as a default for one-off inspection; the overnight single-GPU batch uses `MODELS_TO_RUN` instead.


In [ ]:
AVAILABLE_MODELS = {
    "qwen-1.5b": "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
    "qwen-7b":   "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",
    "llama-8b":  "deepseek-ai/DeepSeek-R1-Distill-Llama-8B",
    # Phase 2 (need compatibility fix + spot-check first, see ROADMAP.md):
    #   "phi4-reasoning": "microsoft/Phi-4-reasoning",   # needs a ChatML system prompt
    #   "qwen3-8b":       "Qwen/Qwen3-8B",               # needs enable_thinking=True
}

# ORDER MATTERS: fastest-to-slowest. If an overnight run doesn't finish
# everything, the smaller/faster models complete ALL datasets before the
# slower ones even start, rather than everything being partially done.
MODELS_TO_RUN = ["qwen-1.5b", "qwen-7b", "llama-8b"]

# All three datasets registered in datasets_registry.py. math500 first
# since it's the most validated / most central to the project so far.
DATASETS_TO_RUN = ["math500", "gsm8k", "aime2024"]

# Items per (model, dataset) pair. Automatically capped at each dataset's
# real pool size (aime2024 only has 30 problems total -- requesting more
# than that just uses all 30, it will not error).
N = 20

print(f"Models this run will cover: {MODELS_TO_RUN}")
print(f"Datasets this run will cover: {DATASETS_TO_RUN}")
print(f"N per (model, dataset) pair: {N} (auto-capped by pool size, e.g. aime2024 -> 30 max)")
print(f"Total (model, dataset) combinations: {len(MODELS_TO_RUN) * len(DATASETS_TO_RUN)}")

# MODEL_KEY / MODEL_NAME below are ONLY used if you switch to the multi-GPU
# path (Section 5A), which runs one model per notebook execution rather than
# looping. Not used by 5B at all -- ignore these two lines unless you're
# specifically using 5A.
MODEL_KEY = "qwen-1.5b"
MODEL_NAME = AVAILABLE_MODELS[MODEL_KEY]

# All three models share the DeepSeek <think>/</think> convention. A
# different family would need harness.py's token handling re-verified
# first (ROADMAP.md, Rule M1).


## 3. Pick the sample

n=40, fixed seed, reproducible.

In [ ]:
import random
from datasets import load_dataset

ds = load_dataset("HuggingFaceH4/MATH-500", split="test")
N = 10
idxs = list(range(len(ds)))
random.Random(7).shuffle(idxs)
all_idxs = idxs[:N]
print(f"selected {len(all_idxs)} MATH-500 problems for the drift run")


## 4. Smoke test (multi-GPU path only)

**Skip this cell entirely if Section 2 said to use the single-GPU path.** This is only useful when there are multiple GPUs to spread across.

In [ ]:
import subprocess, time

def launch(gpu_id, indices, out):
    log = open(f"{out}.log", "w")
    return subprocess.Popen(
        ["python", "drift_run_worker.py", "--gpu", str(gpu_id), "--device", str(gpu_id),
         "--indices", ",".join(map(str, indices)), "--model", MODEL_NAME,
         "--short_budget", "24576", "--long_budget", "24576", "--out", out],
        stdout=log, stderr=subprocess.STDOUT,
    ), log

if N_GPUS >= 2:
    test_procs = []
    for gpu_id, idx in zip([1, 2] if N_GPUS > 2 else [0, 1], all_idxs[:2]):
        p, log = launch(gpu_id, [idx], f"drift_test_gpu{gpu_id}_{RUN_TAG}.json")
        test_procs.append((gpu_id, p, log))
        print(f"launched test on gpu {gpu_id} for idx {idx}")
    while any(p.poll() is None for _, p, _ in test_procs):
        time.sleep(15)
    for gpu_id, p, log in test_procs:
        print(f"gpu {gpu_id}: exit code {p.returncode}  (0 = good)")
    for gpu_id, _, _ in test_procs:
        print(f"\n--- drift_test_gpu{gpu_id}_{RUN_TAG}.json.log ---")
        print(open(f"drift_test_gpu{gpu_id}_{RUN_TAG}.json.log").read())
else:
    print("Only 1 GPU visible -- skipping the multi-GPU smoke test. Go to Section 5B.")


## 5A. Full launch — multi-GPU path

**Only run this if Section 2 detected 2+ GPUs and the Section 4 smoke test passed.** If you're on a single GPU, skip to 5B instead.

In [ ]:
if N_GPUS >= 2:
    WORKER_GPUS = list(range(1, N_GPUS)) if N_GPUS > 2 else [0, 1]
    shards = [all_idxs[i::len(WORKER_GPUS)] for i in range(len(WORKER_GPUS))]

    procs = []
    for gpu_id, shard in zip(WORKER_GPUS, shards):
        if not shard:
            continue
        p, log = launch(gpu_id, shard, f"drift_shard_gpu{gpu_id}_{RUN_TAG}.json")
        procs.append((gpu_id, p, log))
        print(f"launched gpu {gpu_id} with {len(shard)} items")
    print(f"\n{len(procs)} workers running across GPUs {WORKER_GPUS}")
else:
    print("Only 1 GPU -- this cell does nothing. Use Section 5B instead.")


## 5A-check. Progress check (multi-GPU path only)

Safe to re-run any time. Skip if you're on the single-GPU path.

In [ ]:
if N_GPUS >= 2:
    done = sum(1 for _, p, _ in procs if p.poll() is not None)
    print(f"{done}/{len(procs)} workers finished")
    for gpu_id, p, _ in procs:
        print(f"  gpu {gpu_id}: {'done' if p.poll() is not None else 'running'}")
else:
    print("Only 1 GPU -- nothing to check here. See Section 5B's own progress prints.")


## 5B. Full run — single-GPU sequential path (loops over ALL models AND datasets)

**Only run this if Section 2 detected exactly 1 GPU.** For each model in `MODELS_TO_RUN`, loads it ONCE, then runs every dataset in `DATASETS_TO_RUN` against it before moving to the next model (loading a model is expensive; re-loading it per dataset would be wasteful). Good for leaving overnight, unattended.

**Matches Mirage's TTBC1:** short = true natural generation, long = uncapped Wait injection, ceiling = 24576 for every dataset (the ceiling is a property of the MODEL, not the dataset -- see `datasets_registry.py`).

**Built for unattended overnight running, three layers of error handling:**
- A single ITEM failing (e.g. a weird generation edge case) is caught, logged, and skipped -- the rest of that dataset continues.
- A single DATASET failing to even load (e.g. a network hiccup) is caught, logged, and skipped -- the rest of that model's datasets continue.
- A single MODEL failing entirely (e.g. an unrecoverable OOM) is caught, logged, and skipped -- the run moves on to the next model rather than stopping for the night.

None of this papers over real problems -- every error is printed with its type and message so you can review what happened in the morning. It just means one bad item/dataset/model doesn't sacrifice everything else that could still complete.

**Resume support is per (model, dataset) pair.** Each combination saves to its own `drift_run_results_singlegpu_<model>_<dataset>.json` after every item. Re-running this cell picks up exactly where it left off, at whatever granularity it stopped.

**Realistic time expectation, worth knowing before this runs unattended:** based on llama-8b's observed ~9 min/generation on MATH-500, llama-8b alone at N=20 across all 3 datasets could take on the order of 18 hours. This will very likely NOT finish everything in one night. `MODELS_TO_RUN` is ordered fastest-to-slowest specifically so whatever DOES complete is the most useful subset (full coverage on the faster models, rather than partial coverage everywhere). Nothing is lost either way -- resume picks up next time.

In [ ]:
import json, time, gc, os, shutil, random
from transformers import AutoModelForCausalLM, AutoTokenizer
from harness import generate_natural
from wait_injection import generate_stop_triggered_wait
from datasets_registry import load_one
from drift_run_worker import categorize  # reuse the already-tested categorization logic

SHORT_BUDGET = 24576   # short = natural generation, this is just a backstop,
                       # uniform across datasets, rarely actually hit
# LONG budget is now PER-DATASET (see datasets_registry.py): math500 and
# gsm8k are capped at 16384 (a disclosed deviation from strict TTBC1
# fidelity, for compute-time reasons), aime2024 keeps the full 24576
# ceiling. Pulled from cfg["long_budget"] in run_one_item below, not a
# flat constant here.


def clear_model_disk_cache(model_name):
    """Delete a finished model's downloaded weights from the HF cache, so
    running several models back to back doesn't exhaust disk space on a
    later download (this crashed llama-8b's download once already)."""
    cache_dir = os.path.expanduser("~/.cache/huggingface/hub")
    folder = "models--" + model_name.replace("/", "--")
    path = os.path.join(cache_dir, folder)
    if os.path.exists(path):
        shutil.rmtree(path)
        print(f"  cleared disk cache: {path}")


def run_one_item(model, tokenizer, cfg, item, seed=0):
    """Run short+long generation and grading for a single item. Lets
    exceptions propagate -- the caller (run_one_dataset) wraps this in the
    per-item try/except, so a bad item is skipped, not fatal."""
    q = item[cfg["prompt_field"]]
    gold = cfg["extract_gold"](item)

    t0 = time.time()
    short_text, short_think, _, _, _ = generate_natural(
        model, tokenizer, q, max_new_tokens=SHORT_BUDGET, seed=seed, output_scores=False)
    dt_short = time.time() - t0
    short_pred = cfg["extract_pred"](short_text)
    short_correct, short_tier = cfg["is_equiv"](short_pred, gold)
    short_closed = "</think>" in short_text

    t0 = time.time()
    long_text, long_think, _, _, _, n_inj, inj_offsets = generate_stop_triggered_wait(
        model, tokenizer, q, long_budget=cfg["long_budget"], max_injections=None,
        seed=seed, output_scores=False)
    dt_long = time.time() - t0
    long_pred = cfg["extract_pred"](long_text)
    long_correct, long_tier = cfg["is_equiv"](long_pred, gold)

    category = categorize(short_pred, short_correct, long_pred, long_correct)

    return dict(
        gold=gold, category=category,
        short_pred=short_pred, short_correct=short_correct, short_tier=short_tier,
        short_think_tokens=short_think, short_closed_naturally=short_closed,
        short_wall_clock_sec=round(dt_short, 1), short_text=short_text,
        long_pred=long_pred, long_correct=long_correct, long_tier=long_tier,
        long_think_tokens=long_think, long_n_injections=n_inj,
        long_injection_offsets=inj_offsets, long_wall_clock_sec=round(dt_long, 1),
        long_text=long_text,
    )


def run_one_dataset(model, tokenizer, model_key, dataset_key, n_requested):
    """Runs one dataset against an already-loaded model. Per-item errors are
    caught and skipped here, so one bad item never stops the rest."""
    ds, cfg = load_one(dataset_key)
    pool_size = cfg["pool_size"]
    n = min(n_requested, pool_size, len(ds))

    idxs = list(range(len(ds)))
    random.Random(7).shuffle(idxs)
    sel_idxs = idxs[:n]

    run_tag = f"{model_key}_{dataset_key}"
    out_path = f"drift_run_results_singlegpu_{run_tag}.json"

    try:
        results = json.load(open(out_path))
        done_idxs = {r["row_idx"] for r in results}
        print(f"  [{run_tag}] resuming: {len(results)} already done, skipping those.")
    except FileNotFoundError:
        results = []
        done_idxs = set()
        cap_note = f" (capped from requested {n_requested} by pool size {pool_size})" if n < n_requested else ""
        print(f"  [{run_tag}] starting fresh, n={n}{cap_note}.")

    remaining = [i for i in sel_idxs if i not in done_idxs]
    for rank, i in enumerate(remaining):
        try:
            item = ds[i]
            row = run_one_item(model, tokenizer, cfg, item)
            row.update(
                dataset=dataset_key, row_idx=i, model=model_key,
                unique_id=item.get("unique_id") or item.get("ID"),
                subject=item.get("subject"), level=item.get("level"),
            )
            results.append(row)
            with open(out_path, "w") as f:
                json.dump(results, f, indent=2)
            print(f"  [{run_tag}] [{rank + 1}/{len(remaining)}] idx={i} category={row['category']} "
                  f"short={row['short_correct']} long={row['long_correct']} "
                  f"({row['short_wall_clock_sec']:.0f}s+{row['long_wall_clock_sec']:.0f}s)")
        except Exception as e:
            # ITEM-LEVEL catch: log and move to the next item, don't lose the rest.
            print(f"  [{run_tag}] idx={i} ERROR (skipping this item): {type(e).__name__}: {e}")
            continue

    print(f"  [{run_tag}] dataset done: {len(results)}/{n}")
    return results


def load_model_with_retry(model_name, max_attempts=2):
    """Loads a model, and if it fails with an OSError (the exact error seen
    when a cache is corrupted or a download was interrupted -- HF's own
    message literally says "make sure you don't have a local directory with
    the same name"), clears that model's disk cache and retries once with a
    forced fresh download. Without the retry, a corrupted cache from an
    earlier interrupted attempt just fails the same way every time."""
    last_err = None
    for attempt in range(1, max_attempts + 1):
        try:
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            model = AutoModelForCausalLM.from_pretrained(
                model_name, torch_dtype=torch.bfloat16, device_map="cuda:0"
            )
            model.eval()
            return model, tokenizer
        except OSError as e:
            last_err = e
            print(f"  load attempt {attempt}/{max_attempts} failed: {type(e).__name__}: {e}")
            if attempt < max_attempts:
                print("  clearing (possibly corrupted/incomplete) cache and retrying with a fresh download...")
                clear_model_disk_cache(model_name)
                time.sleep(3)
    raise last_err


def run_one_model(model_key):
    """Loads a model ONCE, runs every dataset in DATASETS_TO_RUN against it,
    then frees GPU and disk before returning. DATASET-LEVEL errors (e.g. a
    dataset failing to load) are caught here so one bad dataset doesn't
    skip the rest of this model's datasets. The load itself is now INSIDE
    the try/finally too -- a load that succeeds late (after a retry) or
    fails outright still gets cleanup, whereas before, a load failure
    happened before the try/finally even started, so the corrupted cache
    was never cleared and a retry would just fail the same way again."""
    model_name = AVAILABLE_MODELS[model_key]
    cleanup_zombie_kernels()

    model, tokenizer = None, None
    try:
        print(f"[{model_key}] loading {model_name} ...")
        model, tokenizer = load_model_with_retry(model_name)
        print(f"[{model_key}] loaded.\n")

        for dk in DATASETS_TO_RUN:
            print(f"--- [{model_key}] dataset: {dk} ---")
            try:
                run_one_dataset(model, tokenizer, model_key, dk, N)
            except Exception as e:
                # DATASET-LEVEL catch: log and move to the next dataset.
                print(f"[{model_key}/{dk}] DATASET ERROR (skipping to next dataset): "
                      f"{type(e).__name__}: {e}")
                continue
    finally:
        # Always free GPU + disk, even if loading itself failed after all
        # retries (model/tokenizer may be None in that case -- del on a
        # possibly-unset name would itself raise, so check first).
        if model is not None:
            del model
        if tokenizer is not None:
            del tokenizer
        gc.collect()
        torch.cuda.empty_cache()
        clear_model_disk_cache(model_name)
        print(f"[{model_key}] all datasets attempted (or load failed), GPU and disk freed.\n")


if N_GPUS < 2:
    for mk in MODELS_TO_RUN:
        print("=" * 70)
        print(f"MODEL: {mk}")
        print("=" * 70)
        try:
            run_one_model(mk)
        except Exception as e:
            # MODEL-LEVEL catch: log and move to the next model. This is the
            # outermost safety net -- nothing should be able to end the
            # whole overnight run early because of one bad model.
            print(f"[{mk}] MODEL-LEVEL FATAL ERROR (skipping to next model): "
                  f"{type(e).__name__}: {e}")
            try:
                gc.collect()
                torch.cuda.empty_cache()
            except Exception:
                pass
            continue
    print("All models/datasets in this run attempted. Check the printed log above for any skipped items/datasets/models.")
else:
    print("2+ GPUs visible -- use Section 5A instead, this cell does nothing.")


## 6. Merge and report — the actual result

Works with either path's output. **The key number is drift vs reverse.** Truncations are reported as their own honest category.

In [ ]:
import json
from collections import Counter

def report_one(run_tag, label):
    """Load and summarize one model's results. Works for both the single-GPU
    loop output and the multi-GPU shard output."""
    all_results = []
    if N_GPUS >= 2:
        for gpu_id, p, log in procs:
            try:
                all_results.extend(json.load(open(f"drift_shard_gpu{gpu_id}_{run_tag}.json")))
            except (FileNotFoundError, json.JSONDecodeError):
                pass
    else:
        try:
            all_results = json.load(open(f"drift_run_results_singlegpu_{run_tag}.json"))
        except FileNotFoundError:
            print(f"[{label}] no results file yet.\n")
            return None

    n = len(all_results)
    if n == 0:
        print(f"[{label}] no results yet.\n")
        return None

    with open(f"drift_run_results_{run_tag}.json", "w") as f:
        json.dump(all_results, f, indent=2)

    cats = Counter(r["category"] for r in all_results)
    n_drift, n_reverse = cats.get("drift", 0), cats.get("reverse", 0)
    n_judged = n_drift + n_reverse + cats.get("stable_correct", 0) + cats.get("stable_wrong", 0)
    n_trunc = cats.get("short_truncated", 0) + cats.get("long_truncated", 0)

    print(f"===== {label} =====  ({n} items)")
    for cat in ["drift", "reverse", "stable_correct", "stable_wrong", "short_truncated", "long_truncated"]:
        print(f"  {cat:16s}: {cats.get(cat, 0)}")
    if n_judged:
        print(f"  -> of {n_judged} judged: drift {n_drift} ({n_drift/n_judged:.0%}), "
              f"reverse {n_reverse} ({n_reverse/n_judged:.0%})")
    print(f"  -> truncated: {n_trunc}/{n} ({n_trunc/n:.0%})")
    if n_drift > n_reverse:
        print("  -> drift > reverse: consistent with replication.\n")
    elif n_drift == n_reverse:
        print("  -> drift == reverse: no directional signal.\n")
    else:
        print("  -> reverse > drift: unexpected, worth a look.\n")
    return all_results

# Report every (model, dataset) combination that was run.
per_combo = {}
for mk in MODELS_TO_RUN:
    for dk in DATASETS_TO_RUN:
        rt = f"{mk}_{dk}"
        res = report_one(rt, f"{mk} on {dk}")
        if res is not None:
            per_combo[(mk, dk)] = res

print(f"Combinations with results so far: {len(per_combo)} of {len(MODELS_TO_RUN) * len(DATASETS_TO_RUN)}")

# `all_results` = the LAST combination reported, so Sections 7 and 8 below
# (hand-check, drift-case listing) operate on it. To inspect a specific
# combination there, set e.g.:
#   all_results = per_combo[("qwen-7b", "gsm8k")]
last_combo = list(per_combo.keys())[-1] if per_combo else None
all_results = per_combo[last_combo] if last_combo else []
if last_combo:
    print(f"(Sections 7/8 below default to showing: {last_combo})")


## 7. Hand-check ambiguous matcher calls

In [ ]:
to_check = [r for r in all_results
            if (r["short_tier"] == "no_match" and r["short_pred"] is not None)
            or (r["long_tier"] == "no_match" and r["long_pred"] is not None)]
print(f"{len(to_check)} items with a non-truncation no_match tier:\n")
for r in to_check:
    print(f"idx={r['row_idx']} lvl={r['level']} category={r['category']} gold={r['gold']!r}")
    print(f"   short: pred={r['short_pred']!r} tier={r['short_tier']}")
    print(f"   long:  pred={r['long_pred']!r} tier={r['long_tier']}\n")


## 8. List the actual drift cases

In [ ]:
drift_cases = [r for r in all_results if r["category"] == "drift"]
print(f"{len(drift_cases)} drift cases:\n")
for r in drift_cases:
    print(f"idx={r['row_idx']} lvl={r['level']} subj={r['subject']} gold={r['gold']!r}")
    print(f"   short (correct): {r['short_pred']!r}")
    print(f"   long (wrong):    {r['long_pred']!r}  ({r['long_n_injections']} injections)")
    print(f"   ...long tail: {r['long_text'][-300:]}\n")
